# 🧬 17 — Single-particle reconstruction from unknown views

**Audience:** biology student moving toward research · **Time:** ~4 hours · **Hardware:** free Colab CPU · **Framework:** NumPy · **Prerequisites:** 00–08

**Goals:** explain the concept, run and modify the toy model, identify one failure mode, and connect the method to an original paper. All data are synthetic and for research and education.


## 1. Biological and computational motivation

A purified protein appears in many orientations on the grid. Each low-dose particle image is a 2D projection, modulated by the microscope and noise. Single-particle analysis estimates which orientations go with which images, groups unwanted or different particles, and combines the information into a 3D map. Unlike tomography, the views come from many copies rather than repeated tilts of one region. That can reach fine detail when copies are sufficiently similar, but it loses their original cellular positions.


## 2. Theory and assumptions

The forward relation is `y_i = H_i P_{R_i}(V) + ε_i`. Here the orientation `R_i` is unknown and must be inferred jointly with the map `V`. In the ideal projection model, the 2D Fourier transform of each projection samples one central plane of the 3D Fourier volume. A diverse orientation distribution fills more of Fourier space. Preferred orientations leave anisotropic gaps even when the particle count is large. Contrast-transfer zeros, motion, and conformational mixtures add further challenges.


## 3. Runnable experiment

Predict a result before running. In Colab, NumPy is included; elsewhere install it with `%pip install numpy` if needed.


In [ ]:
import numpy as np
SEED=7
print("NumPy",np.__version__,"| device: CPU | seed:",SEED)


In [ ]:
n=17
z,y,x=np.mgrid[-8:9,-8:9,-8:9]
volume=np.exp(-((x-3)**2+y*y+z*z)/8)+.7*np.exp(-((x+3)**2+(y-2)**2+(z+1)**2)/6)
xy=volume.sum(axis=0)   # project along z
yz=volume.sum(axis=2)   # project along x
xz=volume.sum(axis=1)   # project along y
print('3D shape',volume.shape,'2D view shapes',xy.shape,yz.shape,xz.shape)
print('mass in each view',*[round(a.sum(),2) for a in (xy,yz,xz)])
# An axis-aligned backprojection is a crude localization, not a cryo-EM reconstruction.
back=(xy[None,:,:]*yz[:,:,None]*xz[:,None,:])
peak=np.unravel_index(np.argmax(back),back.shape)
print('crude backprojection peak (z,y,x):',peak)
assert np.allclose([xy.sum(),yz.sum(),xz.sum()],volume.sum())


## 4. Interpretation and challenge

All three views have the same total mass but different patterns. The toy “backprojection” uses known, orthogonal axes and omits CTF, noise, normalization, continuous rotations, and iterative refinement. It cannot resolve arbitrary structures. Its purpose is to show why multiple views provide 3D constraints.

**Change one thing:** Change the volume to contain two equal blobs at mirrored z positions. Which projection cannot tell them apart? Remove one view and explain which coordinate becomes ambiguous. Then predict what a strongly preferred orientation would do to a reconstruction.


## 5. Paper bridge

Read the imaging and orientation sections of [a cryo-EM reconstruction review](https://doi.org/10.1016/j.molcel.2015.02.019), then connect the particle-selection problem in [Shi et al.](https://doi.org/10.1007/s40484-019-0191-8) and [CryoMAE](https://openaccess.thecvf.com/content/WACV2025/papers/Xu_CryoMAE_Few-Shot_Cryo-EM_Particle_Picking_with_Masked_Autoencoders_WACV_2025_paper.pdf).

Record the paper’s input, output, supervision, split unit, central assumption, strongest comparison, and one alternate explanation.
